# Taxonomy experimentation
This notebook contains the experiments regarding the detection of datasets from the papers

In [1]:
import sys
sys.path.append('/home/jovyan/BenchmarkingML4KGE_extraction')

from utils.XMLParser import XMLParser
from utils.grobid_service import GrobidService
from utils import experimentation_utils
import json
from tqdm.auto import tqdm
from bert_score import score
import torch
import time
from gliner import GLiNER
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme()

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, StratifiedKFold, LeaveOneOut, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline, Pipeline

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

import re
import copy

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk import sent_tokenize
nltk.download('punkt_tab')

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from datasets import Dataset
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW


with open('../data/dataset_con_rutas_xml.json', 'r', encoding='utf-8') as f:
    kge_dataset=json.load(f)

service=GrobidService()


/home/jovyan/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jovyan/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data.

## Model 1: Embedding model + classifier


In [2]:
def create_chunks(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [3]:
import os
from pathlib import Path
from collections import Counter
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

X_chunks=[]
Y_chunks=[]
paper_ids=[]

for i,paper in enumerate(kge_dataset):
    category=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if text and category:
        fragments=create_chunks(text)
        for frag in fragments:
            X_chunks.append(frag)
            Y_chunks.append(category)
            paper_ids.append(i)
            
print(f"Generated a total of {len(X_chunks)}")


X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X_chunks, Y_chunks, paper_ids, test_size=0.2, random_state=42, stratify=Y_chunks
)
# 4. Definir el Pipeline
# Usamos stop_words para limpiar ruido y subimos ngram_range para capturar conceptos compuestos (ej: "machine learning")
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english', 
        max_features=10000, 
        ngram_range=(1, 2)
    )),
    ('clf', LinearSVC(C=1.0, class_weight='balanced', random_state=42))
])

init_time=time.time()
# 5. Entrenar
pipeline.fit(X_train, y_train)

y_pred_chunks = pipeline.predict(X_test)

# 2. Agrupamos predicciones por Paper ID
paper_votes = {}
paper_ground_truth = {}

for i in range(len(ids_test)):
    p_id = ids_test[i]
    pred = y_pred_chunks[i]
    real = y_test[i]
    
    if p_id not in paper_votes:
        paper_votes[p_id] = []
        paper_ground_truth[p_id] = real
    
    paper_votes[p_id].append(pred)
end_time=time.time()
tiempo_promedio=end_time-init_time
# 3. Calculamos el ganador (Voto Mayoritario) por paper
final_paper_preds = []
final_paper_real = []

for p_id, votes in paper_votes.items():
    voto_ganador = Counter(votes).most_common(1)[0][0]
    final_paper_preds.append(voto_ganador)
    final_paper_real.append(paper_ground_truth[p_id])

# F1 Score a nivel de Fragmento
f1_chunks = f1_score(y_test, y_pred_chunks, average='weighted')

# F1 Score a nivel de Paper Completo (El que te interesa)
f1_papers = f1_score(final_paper_real, final_paper_preds, average='weighted')

print(f"--- RESULTADOS FINALES ---")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"F1-Score (Nivel Fragmentos): {f1_chunks:.4f}")
print(f"F1-Score (Nivel Papers Completos): {f1_papers:.4f}")
print("\nReporte Detallado por Paper:")
print(classification_report(final_paper_real, final_paper_preds))

Generated a total of 1408
--- RESULTADOS FINALES ---
⏱️ Tiempo promedio por paper: 0.7581 seg
F1-Score (Nivel Fragmentos): 0.5861
F1-Score (Nivel Papers Completos): 0.6820

Reporte Detallado por Paper:
                                        precision    recall  f1-score   support

External extra information outside KGs       0.63      0.63      0.63        19
  Internal side information inside KGs       0.83      0.65      0.73        23
                          Other models       0.60      0.50      0.55        12
              Semantic matching models       0.79      0.79      0.79        19
                    Translation models       0.47      0.88      0.61         8

                              accuracy                           0.68        81
                             macro avg       0.66      0.69      0.66        81
                          weighted avg       0.70      0.68      0.68        81



## Only abstract


In [7]:
import os
from pathlib import Path
from collections import Counter
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

X_chunks=[]
Y_chunks=[]
paper_ids=[]

for i,paper in enumerate(kge_dataset):
    category=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if text and category:
        fragments=create_chunks(text)
        for frag in fragments:
            X_chunks.append(frag)
            Y_chunks.append(category)
            paper_ids.append(i)
            
print(f"Generated a total of {len(X_chunks)}")


X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X_chunks, Y_chunks, paper_ids, test_size=0.2, random_state=42, stratify=Y_chunks
)
# 4. Definir el Pipeline
# Usamos stop_words para limpiar ruido y subimos ngram_range para capturar conceptos compuestos (ej: "machine learning")
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english', 
        max_features=10000, 
        ngram_range=(1, 2)
    )),
    ('clf', LinearSVC(C=1.0, class_weight='balanced', random_state=42))
])

init_time=time.time()
# 5. Entrenar
pipeline.fit(X_train, y_train)

y_pred_chunks = pipeline.predict(X_test)

# 2. Agrupamos predicciones por Paper ID
paper_votes = {}
paper_ground_truth = {}

for i in range(len(ids_test)):
    p_id = ids_test[i]
    pred = y_pred_chunks[i]
    real = y_test[i]
    
    if p_id not in paper_votes:
        paper_votes[p_id] = []
        paper_ground_truth[p_id] = real
    
    paper_votes[p_id].append(pred)
end_time=time.time()
tiempo_promedio=end_time-init_time
# 3. Calculamos el ganador (Voto Mayoritario) por paper
final_paper_preds = []
final_paper_real = []

for p_id, votes in paper_votes.items():
    voto_ganador = Counter(votes).most_common(1)[0][0]
    final_paper_preds.append(voto_ganador)
    final_paper_real.append(paper_ground_truth[p_id])

# F1 Score a nivel de Fragmento
f1_chunks = f1_score(y_test, y_pred_chunks, average='weighted')

# F1 Score a nivel de Paper Completo (El que te interesa)
f1_papers = f1_score(final_paper_real, final_paper_preds, average='weighted')

print(f"--- RESULTADOS FINALES ---")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"F1-Score (Nivel Fragmentos): {f1_chunks:.4f}")
print(f"F1-Score (Nivel Papers Completos): {f1_papers:.4f}")
print("\nReporte Detallado por Paper:")
print(classification_report(final_paper_real, final_paper_preds))

Generated a total of 88
--- RESULTADOS FINALES ---
⏱️ Tiempo promedio por paper: 0.0257 seg
F1-Score (Nivel Fragmentos): 0.3122
F1-Score (Nivel Papers Completos): 0.3122

Reporte Detallado por Paper:
                                        precision    recall  f1-score   support

External extra information outside KGs       0.00      0.00      0.00         4
  Internal side information inside KGs       0.30      0.60      0.40         5
                          Other models       0.00      0.00      0.00         3
              Semantic matching models       0.67      0.50      0.57         4
                    Translation models       1.00      0.50      0.67         2

                              accuracy                           0.33        18
                             macro avg       0.39      0.32      0.33        18
                          weighted avg       0.34      0.33      0.31        18



## Sectioned

In [8]:
import os
from pathlib import Path
from collections import Counter
resultados = []
tiempos = []
scores_f1 = []

base_path = Path(os.getcwd()).parent

X_chunks=[]
Y_chunks=[]
paper_ids=[]

for i,paper in enumerate(kge_dataset):
    category=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if text and category:
        fragments=create_chunks(text)
        for frag in fragments:
            X_chunks.append(frag)
            Y_chunks.append(category)
            paper_ids.append(i)
            
print(f"Generated a total of {len(X_chunks)}")


X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X_chunks, Y_chunks, paper_ids, test_size=0.2, random_state=42, stratify=Y_chunks
)
# 4. Definir el Pipeline
# Usamos stop_words para limpiar ruido y subimos ngram_range para capturar conceptos compuestos (ej: "machine learning")
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english', 
        max_features=10000, 
        ngram_range=(1, 2)
    )),
    ('clf', LinearSVC(C=1.0, class_weight='balanced', random_state=42))
])

init_time=time.time()
# 5. Entrenar
pipeline.fit(X_train, y_train)

y_pred_chunks = pipeline.predict(X_test)

# 2. Agrupamos predicciones por Paper ID
paper_votes = {}
paper_ground_truth = {}

for i in range(len(ids_test)):
    p_id = ids_test[i]
    pred = y_pred_chunks[i]
    real = y_test[i]
    
    if p_id not in paper_votes:
        paper_votes[p_id] = []
        paper_ground_truth[p_id] = real
    
    paper_votes[p_id].append(pred)
end_time=time.time()
tiempo_promedio=end_time-init_time
# 3. Calculamos el ganador (Voto Mayoritario) por paper
final_paper_preds = []
final_paper_real = []

for p_id, votes in paper_votes.items():
    voto_ganador = Counter(votes).most_common(1)[0][0]
    final_paper_preds.append(voto_ganador)
    final_paper_real.append(paper_ground_truth[p_id])

# F1 Score a nivel de Fragmento
f1_chunks = f1_score(y_test, y_pred_chunks, average='weighted')

# F1 Score a nivel de Paper Completo (El que te interesa)
f1_papers = f1_score(final_paper_real, final_paper_preds, average='weighted')

print(f"--- RESULTADOS FINALES ---")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"F1-Score (Nivel Fragmentos): {f1_chunks:.4f}")
print(f"F1-Score (Nivel Papers Completos): {f1_papers:.4f}")
print("\nReporte Detallado por Paper:")
print(classification_report(final_paper_real, final_paper_preds))

Generated a total of 128
--- RESULTADOS FINALES ---
⏱️ Tiempo promedio por paper: 0.0544 seg
F1-Score (Nivel Fragmentos): 0.4845
F1-Score (Nivel Papers Completos): 0.4822

Reporte Detallado por Paper:
                                        precision    recall  f1-score   support

External extra information outside KGs       0.50      0.40      0.44         5
  Internal side information inside KGs       0.40      0.40      0.40         5
                          Other models       0.00      0.00      0.00         3
              Semantic matching models       0.75      0.60      0.67         5
                    Translation models       0.75      1.00      0.86         3

                              accuracy                           0.48        21
                             macro avg       0.48      0.48      0.47        21
                          weighted avg       0.50      0.48      0.48        21



# Model 2: Qwen3:1.7B

In [9]:
import os
from ollama import Client
from utils import llm_preprocessing
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
max_context_tokens = 32768 - 2048



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Full paper

In [14]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []
preds=[]
labels=[]

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])

    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)
print(classification_report(labels,preds))

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.6090 seg
🎯 BERTScore F1 promedio: 0.5201
                                        precision    recall  f1-score   support

External extra information outside KGs       0.00      0.00      0.00        16
  Internal side information inside KGs       0.00      0.00      0.00        14
                Other KGC Technologies       0.00      0.00      0.00         0
                          Other models       0.00      0.00      0.00         9
               Semantic matching model       0.00      0.00      0.00         0
              Semantic matching models       0.00      0.00      0.00        12
                    Translation models       0.67      0.20      0.31        10

                              accuracy                           0.03        61
                             macro avg       0.10      0.03      0.04        61
                          weighted avg       0.11      0.03      0.05        61



/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.loca

## Abstract

In [16]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

preds=[]
labels=[]

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)
print(classification_report(labels,preds))

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.1771 seg
🎯 BERTScore F1 promedio: 0.5530
                                        precision    recall  f1-score   support

External extra information outside KGs       0.00      0.00      0.00        15
  Internal side information inside KGs       0.00      0.00      0.00        18
                Other KGC Technologies       0.00      0.00      0.00         0
                          Other models       0.00      0.00      0.00         9
               Semantic matching model       0.00      0.00      0.00         0
              Semantic matching models       0.00      0.00      0.00        14
                    Translation models       1.00      0.12      0.22         8

                              accuracy                           0.02        64
                             macro avg       0.14      0.02      0.03        64
                          weighted avg       0.12      0.02      0.03        64



/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.loca

## Sections

In [17]:
import os
from pathlib import Path
resultados = []
tiempos = []
scores_f1 = []

labels=[]
preds=[]

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)
print(classification_report(labels,preds))

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.1440 seg
🎯 BERTScore F1 promedio: 0.3620
                                        precision    recall  f1-score   support

External extra information outside KGs       0.00      0.00      0.00      11.0
  Internal side information inside KGs       0.00      0.00      0.00      13.0
                                 KeGNN       0.00      0.00      0.00       0.0
                Other KGC Technologies       0.00      0.00      0.00       0.0
                          Other models       0.00      0.00      0.00       4.0
               Semantic matching model       0.00      0.00      0.00       0.0
              Semantic matching models       0.00      0.00      0.00       5.0
                    Translation models       0.00      0.00      0.00       3.0

                              accuracy                           0.00      36.0
                             macro avg       0.00      0.00      0.00      36.0
               

/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/.loca

# Model 3: Llama 3

In [8]:
import os
from pathlib import Path
import ollama
from ollama import Client, ResponseError

resultados = []
tiempos = []
scores_f1 = []
preds=[]
labels=[]

model_name='llama3'
client= Client(timeout=600.0)

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_full_text()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")

    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 2.1139 seg
🎯 BERTScore F1 promedio: 0.9358


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

## Abstract

In [11]:
import os
from pathlib import Path
from ollama import Client, ResponseError

resultados = []
tiempos = []
scores_f1 = []
preds=[]
labels=[]

model_name='llama3'
client=Client(timeout=600.0)

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    text=parser.get_abstract()

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")
    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)
print(classification_report(labels,preds))

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 0.4144 seg
🎯 BERTScore F1 promedio: 0.9342


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

## Sections

In [2]:
import os
from pathlib import Path
from ollama import Client, ResponseError
resultados = []
tiempos = []
scores_f1 = []
preds=[]
labels=[]

model_name='llama3'
client = Client(timeout=600.0)

model_context = '''The semantic matching models and The translation models only use the structure information of internal facts in KGs. The semantic matching models generally use semantic matching-based scoring functions and further consists of tensor/matrix factorization models and neural network models. The translation models apply distance-based scoring functions.
    While Internal side information inside KGs and External extra information outside KGs outside KGs cooperate with additional information (the inside or outside information of KGs except for the structure information) to achieve KGC. Internal side information inside KGs involved in KGs, including node attributes information, entity-related information, relation-related information, neighborhood information, relational path information; External extra information outside KGs outside KGs, mainly including two aspects: rule-based KGC and third-party data sources-based KGC. 
    And if it is not any of the previous models, then it is Other KGC technologies.'''
question = "Given the model definitions mentioned before, choose one of the following taxonomy as the taxonomy of the model mentioned in this paper: Semantic matching model, Translation models, Internal side information inside KGs model, External extra information outside KGs or Other KGC Technologies ? "
base_path = Path(os.getcwd()).parent

for paper in tqdm(kge_dataset, desc="Processing"):
    gt=paper.get('category',[])
    xml_path=paper.get('xml_file')
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    sections=parser.get_sections(target_sections=['Methodology','Model','Embedding'])
    text='\n'.join(sections.values())

    if not text or not gt:
        continue
        print('An error occurred while processing')

    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt = (
           f"Now, given this context for model taxonomy: {model_context} Answer this question: {question} "
           +"Give back the answer only and only in a correct Python list format, for example: ['A']. If you don't know the answer, just return an empty list."

        )
    chat.append({"role":"user","content":prompt})

    init_time=time.time()

    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")

    if len(predictions)==1:
        preds.append(predictions[0])
        labels.append(gt)

    end_time=time.time()
    total_time=end_time-init_time
    tiempos.append(total_time)

    f1_score=experimentation_utils.calcular_bertscore_listas(predictions,gt)
    scores_f1.append(f1_score)

    resultados.append({
            "title": paper.get('title'),
            "time": total_time,
            "bertscore_f1": f1_score,
        })

tiempo_promedio = sum(tiempos) / len(tiempos) if tiempos else 0
score_promedio = sum(scores_f1) / len(scores_f1) if scores_f1 else 0

print("\n" + "="*30)
print("📊 RESULTADOS DEL EXPERIMENTO")
print(f"⏱️ Tiempo promedio por paper: {tiempo_promedio:.4f} seg")
print(f"🎯 BERTScore F1 promedio: {score_promedio:.4f}")
print("="*30)
print(classification_report(labels,preds))

Processing:   0%|          | 0/89 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


📊 RESULTADOS DEL EXPERIMENTO
⏱️ Tiempo promedio por paper: 1.0887 seg
🎯 BERTScore F1 promedio: 0.9328


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.